- python -m venv myenv
- source ./myenv/bin/activate
- pip install ipykernel
- python -m ipykernel install --user --name myenv --display-name "(myenv)"


- github repository 생성
- .gitignore 생성
    - echo ".Trash-0/" >> .gitignore
    - echo "myenv/" >> .gitignore
- git init
- git config --global --add safe.directory /workspace
- git remote add origin [ssh repository address]
- git config --global user.email [github email address]
- git config --global user.name [github name]

- git add .
- git commit -m "Initial commit"
- git branch
    - 이걸 해서 main이면 넘어가고, 아니면 아래 명령어 수행
    - git branch -M main
- git push -u origin main
    - ssh key 에러가 뜬다면 아래 명령어 수행
- ssh-keygen -t ed25519 -C "본인의 깃허브 이메일"
- cat ~/.ssh/id_ed25519.pub
    - 위 명령어로 나오는 걸 복사해서 github로 이동. ssh-ed로 시작하며 그대로 복사하면 됨
        - ssh-ed25519 ...
    - 로그인 > 우측 상단 프로필 > Settings > 왼쪽 메뉴 SSH and GPG keys 클릭
    - 초록색 버튼 New SSH key 클릭
    - Title에 "scnu jupyter server"dlqfur
    - key 붙여 넣고 Add SSH key 클릭
- git push -u origin main

# 0. 데이터 다운로드 명령어

- prompt1: 지금 업로드한 데이터는 aihub에서 자료를 다운받기 위해 작성한 json 파일이야. 이 구조를 잘 보면 어떻게 다운로드 shell 명령어를 만들지 알 수 있어. 다음 문법을 이해하고 준비되면 'ok'라고 말해줘

1. 전체 데이터 조회 옵션 -mode l
 - aihubshell -mode l 
2. datasetkey를 이용한 조회
 - aihubshell -mode l -datasetkey {datasetkey 실제값}
3. filekey를 이용한 자료 다운로드
 - aihubshell -mode d -datasetkey {datasetkey 실제값} -filekey {다운로드할 파일키}
4. 여러 filekey를 이용한 자료 다운로드
 - aihubshell -mode d -datasetkey {datasetkey 실제값} -filekey {다운로드할 파일키1},{다운로드할 파일키2},{다운로드할 파일키3}
 
- prompt2: 이번엔 Training 데이터 중에서 Apple, garlic, mandarine, onion, pear, potato를 모두 다운로드 할 수 있는 shell 명령어를 만들고, 총 용량이 얼마가 되는지 계산해줘

(결과물)
- aihubshell -mode d -datasetkey 149 -filekey 396957,396958,396959,396960,396961,396962,396969,396970,396971,396975,396976,396977,396978,396979,396980,396981,396982,396983,396984,396985,396986,396987,396988,396989,396990,396991,396992,397002,397003,397004,397005,397006,397007,396903,396904,396905,396906,396907,396908,396918,396919,396920,396921,396922,396923,396924,396925,396926,396927,396928,396929,396930,396931,396932,396933,396934,396935,396936,396937,396938,396948,396949,396950,396951,396952,396953

- (결과물) shell 명령어를 수행하고 데이터를 다운로드 함
- 그리고 아래 1.make_train_dataset.py 수행하면 됨.

# 1. make_train_dataset.py

In [9]:
# cate3의 요소를 파악하기 위함

import os
import json
import zipfile
from tqdm import tqdm

def check_cate3_from_zips(base_path):
    """
    압축을 풀지 않고, 라벨링 zip 파일 내부의 JSON을 읽어 cate3 값을 확인합니다.
    """
    # 1. 라벨링 zip 파일들이 모인 폴더 찾기
    label_base_path = ""
    try:
        dirlist = os.listdir(base_path)
        for folder in dirlist:
            if "라벨링" in folder:
                label_base_path = os.path.join(base_path, folder)
                break
    except FileNotFoundError:
        print(f"오류: '{base_path}' 경로를 찾을 수 없습니다.")
        return

    if not label_base_path:
        print("라벨링 폴더를 찾을 수 없습니다.")
        return

    # 2. 폴더 내의 모든 .zip 파일 목록 가져오기
    zip_files = [f for f in os.listdir(label_base_path) if f.endswith('.zip')]
    
    if not zip_files:
        print(f"경로 내에 zip 파일이 없습니다: {label_base_path}")
        return

    cate3_values = set()
    print(f"분석 대상 폴더: {label_base_path}")
    print(f"총 {len(zip_files)}개의 압축 파일을 분석합니다...")

    # 3. 각 Zip 파일을 열어서 내부의 JSON 하나씩 샘플링
    for zip_name in tqdm(zip_files, desc="Zip 파일 분석 중"):
        zip_path = os.path.join(label_base_path, zip_name)
        try:
            with zipfile.ZipFile(zip_path, 'r') as z:
                # zip 안에 있는 파일 목록 중 .json만 추출
                json_in_zip = [f for f in z.namelist() if f.endswith('.json')]
                
                # 효율을 위해 각 zip당 최대 5개까지만 샘플링 (모두 보려면 [:5] 제거)
                for json_file in json_in_zip[:5]:
                    with z.open(json_file) as f:
                        data = json.load(f)
                        if 'cate3' in data:
                            cate3_values.add(data['cate3'])
        except Exception as e:
            continue

    # 4. 결과 출력
    print("\n" + "="*40)
    print(f"📊 분석 결과")
    print(f"발견된 cate3 고유 값: {list(cate3_values)}")
    print("="*40)

# 실행 경로 설정 (사용자님의 에러 메시지에 나온 경로의 상위 경로 입력)
download_folder_name = r"068.농산물_품질(QC)_이미지/01.데이터/1.Training/"
check_cate3_from_zips(download_folder_name)

분석 대상 폴더: 068.농산물_품질(QC)_이미지/01.데이터/1.Training/라벨링데이터_230921_add
총 33개의 압축 파일을 분석합니다...


Zip 파일 분석 중: 100%|██████████| 33/33 [00:00<00:00, 44.32it/s]


📊 분석 결과
발견된 cate3 고유 값: ['특', '보통', '상']


In [10]:
import os
import zipfile
import json
import shutil
from tqdm import tqdm

# 1. 설정
classes = ["apple", "mandarine", "onion", "pear", "potato"]
# classes = ["apple", "mandarine"]
qualities = ["super", "good", "normal"] # 3단계 등급으로 수정
download_folder_name = r"068.농산물_품질(QC)_이미지/01.데이터/1.Training/"
train_folder_name = "train"

# 2. 폴더 구조 생성 (train/apple/super, train/apple/good...)
for class_name in classes:
    for q in qualities:
        os.makedirs(os.path.join(train_folder_name, class_name, q), exist_ok=True)

# 3. 데이터 경로 파악 (원천 및 라벨링 폴더 찾기)
dirlist = os.listdir(download_folder_name)
image_base_path = ""
label_base_path = ""

for folder in dirlist:
    if "원천" in folder:
        image_base_path = os.path.join(download_folder_name, folder)
    elif "라벨링" in folder:
        label_base_path = os.path.join(download_folder_name, folder)

# 4. 클래스별 처리 시작
for class_name in classes:
    print(f"\n[{class_name.upper()}] 처리 및 등급 분류 시작...")
    
    # 해당 클래스 이미지 zip과 라벨 json zip 리스트 확보
    image_zips = [z for z in os.listdir(image_base_path) if class_name.lower() in z.lower()]
    label_zips = [z for z in os.listdir(label_base_path) if class_name.lower() in z.lower()]

    # 임시로 압축을 풀 공간 생성
    temp_extract_path = os.path.join(train_folder_name, f"temp_{class_name}")
    os.makedirs(temp_extract_path, exist_ok=True)

    # A. 이미지와 라벨 압축 해제
    for zips, base_p in [(image_zips, image_base_path), (label_zips, label_base_path)]:
        for zip_name in tqdm(zips, desc=f"Extracting {class_name} files"):
            with zipfile.ZipFile(os.path.join(base_p, zip_name), 'r') as zip_ref:
                zip_ref.extractall(temp_extract_path)

    # B. JSON 내용을 보고 이미지 파일 이동 (품질 분류 핵심 로직)
    all_files = os.listdir(temp_extract_path)
    img_files = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for img_name in tqdm(img_files, desc=f"Classifying {class_name} by quality"):
        img_path = os.path.join(temp_extract_path, img_name)
        json_path = os.path.splitext(img_path)[0] + ".json" # 확장자만 json으로 변경

        if os.path.exists(json_path):
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    meta = json.load(f)
                    cate3_val = meta.get('cate3', '')

                    # 등급 매칭 (AI Hub 표준 규격 기준)
                    if cate3_val in ['특']:
                        target_q = "super"
                    elif cate3_val in ['상']:
                        target_q = "good"
                    else: # '보통' 혹은 그 외
                        target_q = "normal"

                    # 파일 이동
                    dst_path = os.path.join(train_folder_name, class_name, target_q, img_name)
                    shutil.move(img_path, dst_path)
            except Exception as e:
                print(f"파일 처리 오류 ({img_name}): {e}")
        
        # 원본 이미지나 JSON이 남아있다면 삭제 (정리)
        if os.path.exists(img_path): os.remove(img_path)
        if os.path.exists(json_path): os.remove(json_path)

    # 임시 폴더 삭제
    shutil.rmtree(temp_extract_path)

print("\n등급별 데이터 구축이 완료되었습니다!")


[APPLE] 처리 및 등급 분류 시작...


Classifying apple by quality: 100%|██████████| 21896/21896 [00:01<00:00, 20639.01it/s]



[MANDARINE] 처리 및 등급 분류 시작...


Classifying mandarine by quality: 100%|██████████| 21168/21168 [00:01<00:00, 20371.68it/s]



[ONION] 처리 및 등급 분류 시작...


...


[PEAR] 처리 및 등급 분류 시작...


Classifying pear by quality: 100%|██████████| 21168/21168 [00:01<00:00, 19309.29it/s]



[POTATO] 처리 및 등급 분류 시작...


Classifying potato by quality: 100%|██████████| 21168/21168 [00:01<00:00, 15786.78it/s]


등급별 데이터 구축이 완료되었습니다!


`shell`
- 터미널에서 train 폴더 안에 들어가서 각 폴더 및 세부 폴더 내 파일 개수를 세는 리눅스 명령어
- find . -maxdepth 2 -type d -exec sh -c "echo -n '{}: '; find '{}' -type f | wc -l" \;

## 1.1. 에러시 추가로 수행할 코드
- 에러가 난 파일만 다시 작업하는 코드
- 에러가 난 파일의 이름을 직접 변경해야 함
- 보통 에러는 다운로드 과정에서 zip이 망가지거나 하는 상황임. 다시 자료를 다운받고 아래 작업을 수행할 것.

In [ ]:
import os
import zipfile

# 1. 에러가 발생했던 파일명과 해당 클래스 매핑
error_files = {
    "Apple_fuji_M.zip": "apple",
    "Apple_fuji_L.zip": "apple"
}

download_folder_name = r"068.농산물_품질(QC)_이미지/01.데이터/1.Training/"
train_folder_name = "train"

# 2. 원천데이터 경로 탐색
dirlist = os.listdir(download_folder_name)
image_file_path = ""
for directory in dirlist:
    if "원천" in directory:
        image_file_path = os.path.join(download_folder_name, directory)
        break

if not image_file_path:
    print("원천데이터 폴더를 찾을 수 없습니다.")
else:
    print(f"재작업 시작: 총 {len(error_files)}개 파일")
    
    # 3. 지정된 에러 파일들에 대해서만 작업 수행
    for zip_file_name, class_name in error_files.items():
        zip_path = os.path.join(image_file_path, zip_file_name)
        extract_path = os.path.join(train_folder_name, class_name)
        
        # 폴더가 없다면 생성
        os.makedirs(extract_path, exist_ok=True)
        
        if os.path.exists(zip_path):
            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_path)
                    print(f"  - [성공] {zip_file_name} -> {extract_path}")
            except Exception as e:
                print(f"  - [실패] {zip_file_name}: {e}")
                print("    (팁: 파일 용량을 확인해보세요. 다운로드가 덜 되었을 가능성이 높습니다.)")
        else:
            print(f"  - [파일 없음] {zip_path} 경로에 파일이 없습니다.")

print("재작업이 완료되었습니다.")

# 2. sampling_train_dataset.py

In [1]:
import os
import random
import shutil

# 0. 샘플링 개수
num_sampling = 800

# 1. 경로 설정
base_dir = os.getcwd()
print(base_dir)
source_root = os.path.join(base_dir, 'train')
target_root = os.path.join(base_dir, f'train_{num_sampling}')

# 2. train_800 루트 폴더 생성
if not os.path.exists(target_root):
    os.makedirs(target_root)

# 3. train 폴더 내의 모든 하위 디렉토리 찾기
subfolders = [f for f in os.listdir(source_root) if os.path.isdir(os.path.join(source_root, f))]

print(f"발견된 폴더 목록: {subfolders}")

for folder in subfolders:
    # 각 폴더의 원본 경로와 복사될 경로 설정
    source_folder_path = os.path.join(source_root, folder)
    target_folder_path = os.path.join(target_root, folder)
    
    # 4. 대상 폴더 안에 동일한 이름의 하위 폴더 생성 (예: train_500/Apple)
    if not os.path.exists(target_folder_path):
        os.makedirs(target_folder_path)
    
    # 5. 해당 폴더 내의 파일 목록 가져오기
    all_files = [f for f in os.listdir(source_folder_path) if os.path.isfile(os.path.join(source_folder_path, f))]
    
    # 6. 랜덤하게 500개 추출 (파일이 500개 미만이면 전체 선택)
    num_to_sample = min(len(all_files), num_sampling)
    sampled_files = random.sample(all_files, num_to_sample)
    
    print(f"[{folder}] 처리 중: {len(all_files)}개 중 {num_to_sample}개 복사 시작...")
    
    # 7. 파일 복사 실행
    for file_name in sampled_files:
        src_file = os.path.join(source_folder_path, file_name)
        dst_file = os.path.join(target_folder_path, file_name)
        shutil.copy2(src_file, dst_file)

print("-" * 30)
print(f"모든 작업이 완료되었습니다. '{target_root}'를 확인하세요!")

/workspace
발견된 폴더 목록: ['onion', 'potato', 'pear', 'apple', 'mandarine']
[onion] 처리 중: 21800개 중 800개 복사 시작...
[potato] 처리 중: 21168개 중 800개 복사 시작...
[pear] 처리 중: 21168개 중 800개 복사 시작...
[apple] 처리 중: 21896개 중 800개 복사 시작...
[mandarine] 처리 중: 21168개 중 800개 복사 시작...
------------------------------
모든 작업이 완료되었습니다. '/workspace/train_800'를 확인하세요!


# 3. model.py

In [ ]:
# CUDA 12.1~12.5 대응 버전 설치 명령어
# 터미널에서 실행할 것

pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [11]:
import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"GPU 사용 가능 여부: {torch.cuda.is_available()}")
print(f"사용 중인 GPU 장치: {torch.cuda.get_device_name(0)}")

PyTorch 버전: 2.5.1+cu121
GPU 사용 가능 여부: True
사용 중인 GPU 장치: NVIDIA H100 PCIe


In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiTaskFruitModel(nn.Module):
    def __init__(self, num_classes, num_qualities):
        super(MultiTaskFruitModel, self).__init__()
        
        # 1. 공통 특징 추출 레이어 (Shared Backbone)
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1)
        
        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # 2. Class 분류 헤드
        self.class_fc = nn.Linear(256, num_classes)
        
        # 3. Quality 분류 헤드 (중간 특징 결합 구조)
        self.quality_fc1 = nn.Linear(128, 128)
        self.quality_fc2 = nn.Linear(256, 128)
        self.quality_final = nn.Linear(128, num_qualities)

    def forward(self, x):
        # 레이어 1, 2 통과
        x = F.silu(self.conv1(x))
        x = F.silu(self.conv2(x))
        
        # 레이어 3 통과 후 첫 번째 GAP (gap_hidden_1 역할)
        x3 = F.silu(self.conv3(x))
        gap_hidden_1 = self.gap(x3).view(x3.size(0), -1)
        
        # 레이어 4 통과 후 두 번째 GAP (gap_hidden_2 역할)
        x4 = F.silu(self.conv4(x3))
        gap_hidden_2 = self.gap(x4).view(x4.size(0), -1)
        
        # --- Class 출력 ---
        # gap_hidden_2를 사용하여 클래스 분류
        class_out = self.class_fc(gap_hidden_2)
        # CrossEntropyLoss를 사용할 계획이라면 softmax는 생략 가능하지만, 
        # Keras 코드와 동일하게 확률을 반환하려면 아래 줄을 유지하세요.
        class_out = F.softmax(class_out, dim=1)
        
        # --- Quality 출력 ---
        # gap_hidden_1과 gap_hidden_2를 각각 Dense 레이어에 통과
        q_h1 = F.silu(self.quality_fc1(gap_hidden_1))
        q_h2 = F.silu(self.quality_fc2(gap_hidden_2))
        
        # Keras의 Average() 레이어와 동일하게 평균 계산
        quality_hidden = (q_h1 + q_h2) / 2
        
        quality_out = self.quality_final(quality_hidden)
        quality_out = F.softmax(quality_out, dim=1)
        
        return class_out, quality_out


def get_model(classes, qualities):
    num_classes = len(classes)
    num_qualities = len(qualities)
    model = MultiTaskFruitModel(num_classes=num_classes, num_qualities=num_qualities)
    
    # GPU(H100) 환경이 감지되면 모델을 GPU로 이동시킵니다.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    return model

In [4]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class FruitClassQualitier(nn.Module):
#     def __init__(self, num_classes):
#         super(FruitClassifier, self).__init__()
        
#         # 레이어 1: 기초 특징 추출 (직선, 곡선 등)
#         # In: (N, 3, 64, 64) -> Out: (N, 16, 32, 32)
#         self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=2, padding=2)
        
#         # 레이어 2: 복잡한 특징 추출
#         # In: (N, 16, 32, 32) -> Out: (N, 32, 16, 16)
#         self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=2, padding=1)
        
#         # 레이어 3: 더 복잡한 특징 추출
#         # In: (N, 32, 16, 16) -> Out: (N, 32, 8, 8)
#         self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=2, padding=1)
        
#         # 전역 평균 풀링 (Global Average Pooling)
#         self.gap = nn.AdaptiveAvgPool2d(1)
        
#         # 최종 분류 레이어 (확률 반환)
#         self.fc = nn.Linear(32, num_classes)

#     def forward(self, x):
#         # x: [Batch, Channel, Height, Width]
        
#         x = F.silu(self.conv1(x)) # SiLU는 TensorFlow의 Swish와 동일한 함수입니다.
#         x = F.silu(self.conv2(x))
# #         x = F.silu(self.conv3(x))
        
#         x = self.gap(x)           # (N, 32, 1, 1) 형태로 변환
#         x = x.view(x.size(0), -1) # (N, 32)로 평탄화
        
#         # PyTorch의 CrossEntropyLoss를 사용할 경우 마지막에 Softmax를 생략하는 것이 표준이지만,
#         # 기존 설계대로 확률값을 반환하기 위해 Softmax를 명시합니다.
#         x = F.softmax(self.fc(x), dim=1)
        
#         return x

# def get_model(classes):
#     num_classes = len(classes)
#     model = FruitClassQualitier(num_classes)
    
#     # GPU(H100) 환경이 감지되면 모델을 GPU로 이동시킵니다.
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     model.to(device)
    
#     return model

# 4. dataloader.py

In [36]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. 이미지 전처리 정의 (PyTorch 표준 방식)
# numpy 수동 계산 대신 transforms 라이브러리를 사용하면 가독성과 성능이 좋아집니다.
data_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(), # 이미지를 [0, 1] 범위 텐서로 변환하고 (C, H, W) 순서로 바꿈
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # 표준 정규화
])

class AIHubDataset(Dataset):
    def __init__(self, file_list, classes, qualities, transform=None):
        self.file_list = file_list
        self.classes = classes
        self.qualities = qualities
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        
        # 1. 이미지 로드 로직 (기존 예외 처리 유지)
        try:
            img = Image.open(file_name).convert('RGB')
        except Exception as e:
            print(f"로드 실패 {file_name}, 재시도 중...")
            return self.__getitem__(idx - 1 if idx > 0 else idx + 1)

        # 2. 전처리(Transform) 적용
        if self.transform:
            img = self.transform(img)

        # 3. 클래스 라벨 생성 (TensorFlow 코드의 논리 그대로 적용)
        # 예: [1.0, 0.0, 0.0, 0.0, 0.0]
        # os.sep은 윈도우(\), 리눅스(/) 환경에 맞춰 경로를 잘라줍니다.
        parts = file_name.split(os.sep)
        folder_quality = ""
            for q in self.qualities:
                if q.lower() in [p.lower() for p in parts]:
                    folder_quality = q.lower()
                    break

            # 3. 진짜 원-핫 인코딩 생성
            quality = [1.0 if q.lower() == folder_quality else 0.0 for q in self.qualities]

            # 만약 하나도 매칭 안 됐다면? (에러 방지용 디버깅)
            if sum(quality) == 0:
                print(f"Warning: No quality match for {file_name}")
        
        # 5. 최종 반환 (이미지, 클래스 텐서, 품질 텐서)
        return (
            img, 
            torch.tensor(label, dtype=torch.float32), 
            torch.tensor(quality, dtype=torch.float32)
        )
    
def get_dataloader(classes, qualities, train_folder_name, batch_size=32):
    # 모든 이미지 파일 탐색
    file_list = glob.glob(os.path.join(train_folder_name, '**', '*.*'), recursive=True)
    # 이미지 파일 확장자 필터링 (필요시)
    file_list = [f for f in file_list if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    # 데이터셋 인스턴스 생성
    dataset = AIHubDataset(file_list, classes, qualities, transform=data_transforms)
    
    # 데이터로더 생성 (GPU 효율을 위해 shuffle과 num_workers 설정)
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=4, # CPU 코어 수에 맞춰 조절
        pin_memory=True # GPU 전송 속도 향상
    )
    
    return dataloader

IndentationError: unexpected indent (1923954517.py, line 45)

In [5]:
# import os
# import glob
# import torch
# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms
# from PIL import Image

# # 1. 이미지 전처리 정의 (PyTorch 표준 방식)
# # numpy 수동 계산 대신 transforms 라이브러리를 사용하면 가독성과 성능이 좋아집니다.
# data_transforms = transforms.Compose([
#     transforms.Resize((64, 64)),
#     transforms.ToTensor(), # 이미지를 [0, 1] 범위 텐서로 변환하고 (C, H, W) 순서로 바꿈
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # 표준 정규화
# ])

# class AIHubDataset(Dataset):
#     def __init__(self, file_list, classes, qualities, transform=None):
#         self.file_list = file_list
#         self.classes = classes
#         self.transform = transform
#         self.qualities = qualities

#     def __len__(self):
#         return len(self.file_list)

#     def __getitem__(self, idx):
#         file_name = self.file_list[idx]
        
#         try:
#             img = Image.open(file_name).convert('RGB')
#         except Exception as e:
#             # 이미지 로드 실패 시 이전 인덱스 파일 재시도 (기존 로직 유지)
#             print(f"로드 실패 {file_name}, 재시도 중...")
#             return self.__getitem__(idx - 1 if idx > 0 else idx + 1)

#         if self.transform:
#             img = self.transform(img)

#         # 레이어 매칭: 파일 경로에 클래스 이름이 포함되어 있는지 확인
#         # Multi-label 형식으로 반환 (TensorFlow 코드와 동일한 논리)
#         label = [1.0 if class_name.lower() in file_name.lower() else 0.0 for class_name in self.classes]
        
#         quality = ??
        
#         return img, torch.tensor(label, dtype=torch.float32), torch.tensor(quality, dtype=torch.float32)

# def get_dataloader(classes, qualities, train_folder_name, batch_size=32):
#     # 모든 이미지 파일 탐색
#     file_list = glob.glob(os.path.join(train_folder_name, '**', '*.*'), recursive=True)
#     # 이미지 파일 확장자 필터링 (필요시)
#     file_list = [f for f in file_list if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
#     # 데이터셋 인스턴스 생성
#     dataset = AIHubDataset(file_list, classes, qualities, transform=data_transforms)
    
#     # 데이터로더 생성 (GPU 효율을 위해 shuffle과 num_workers 설정)
#     dataloader = DataLoader(
#         dataset, 
#         batch_size=batch_size, 
#         shuffle=True, 
#         num_workers=4, # CPU 코어 수에 맞춰 조절
#         pin_memory=True # GPU 전송 속도 향상
#     )
    
#     return dataloader

# 5. train.py

- 저장한 모델 이름: train_model_5fruits.pth

In [14]:
!nvidia-smi

Thu Feb  5 12:17:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.02              Driver Version: 555.42.02      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 PCIe               Off |   00000000:0D:00.0 Off |                    0 |
| N/A   41C    P0             79W /  350W |    8307MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [15]:
!pip install tqdm

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [37]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm  # 학습 진행률 표시를 위한 라이브러리

# 앞서 정의한 get_model과 get_dataloader가 정의되어 있어야 합니다.
# from model import get_model
# from dataloader import get_dataloader

# 1. 장치 설정 (H100 GPU 사용 가능 여부 확인)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 학습에 사용하는 장치: {device}")

# 2. 클래스 및 폴더 설정
classes = ["apple", "mandarine", "onion", "pear", "potato"]
qualities = ['super', 'good', 'normal']
train_folder_name = 'train'

if not os.path.exists('./models'):
    os.makedirs('./models')

# 3. 모델 및 데이터 로더 초기화
model = get_model(classes, qualities)  # 이미 내부에서 .to(device) 처리가 됨
dataloader = get_dataloader(classes, qualities, train_folder_name, batch_size=32)

# 4. 손실 함수 및 최적화 함수 설정
# TensorFlow의 categorical_crossentropy는 PyTorch의 CrossEntropyLoss와 매칭됩니다.
# (라벨이 One-hot 형태인 경우 BCEWithLogitsLoss가 더 적합할 수 있으나, 기존 설계를 따릅니다.)
# criterion = nn.BCELoss() # 모델의 마지막이 Softmax이므로 Binary Cross Entropy 사용
# 초기 설정 (루프 밖)
best_loss = float('inf')
# Multi-task를 위한 손실 함수 정의 (보통 두 태스크 모두 분류라면 CrossEntropy 사용)
criterion_class = nn.CrossEntropyLoss()
criterion_quality = nn.CrossEntropyLoss()

# optimizer 설정
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. 학습 루프 정의 (TensorFlow의 model.fit 역할)
num_epochs = 1
best_loss = float('inf')

for epoch in range(num_epochs):
    model.train()  # 모델을 학습 모드로 설정
    running_loss = 0.0
    
    # tqdm을 사용하여 진행률 표시
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, labels, quali_labels in pbar:
        # 데이터를 GPU로 이동
        images = images.to(device)
        labels = labels.to(device)
        quali_labels = quali_labels.to(device)
        
        # 가중치 초기화
        optimizer.zero_grad()
        
        # 순전파 (Forward)
        out_class, out_quality = model(images)
        
        # 각각의 loss
        loss_class = criterion_class(out_class, labels)
        loss_quality = criterion_quality(out_quality, quali_labels)
        
        # 5. Total Loss 합산 (두 태스크의 중요도에 따라 가중치를 곱할 수 있습니다)
        # 예: total_loss = loss_class * 0.7 + loss_quality * 0.3
        total_loss = loss_class + loss_quality
        
        # 역전파 및 가중치 업데이트 (Backward & Optimize)
        total_loss.backward()
        optimizer.step()
        
        running_loss += total_loss.item()
        pbar.set_postfix({'loss': running_loss / (pbar.n + 1)})

    # 에폭 종료 후 평균 손실 계산
    epoch_loss = running_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Average Loss: {epoch_loss:.4f}")

    # ModelCheckpoint 기능 구현 (가장 낮은 loss일 때 저장)
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), './models/train_model_5fruits_multitask.pth')
        print(f"  --> Best model saved with loss: {best_loss:.4f}")

print("학습이 완료되었습니다.")

현재 학습에 사용하는 장치: cuda


Epoch 1/1: 100%|██████████| 3348/3348 [11:22<00:00,  4.91it/s, loss=1.91]

Epoch [1/1] Average Loss: 1.9095
  --> Best model saved with loss: 1.9095
학습이 완료되었습니다.


# 6. evaluate.py

- 학습한 모델 train_model_5fruits.pth를 업로드 하여, 학습 데이터에 대해 추론하고 모델의 성능을 평가하는 과정

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm
# 이전에 정의한 클래스와 함수들을 가져옵니다.
# from model import get_model
# from dataloader import get_dataloader

def evaluate_model(classes, qualities, train_folder_name, model_path='./models/train_model_5fruits_multitask.pth'):
    # 1. 장치 설정
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"평가에 사용하는 장치: {device}")

    # 2. 모델 초기화 및 가중치 불러오기
    model = get_model(classes, qualities)
    
    if os.path.exists(model_path):
        # 저장된 state_dict를 불러와 모델에 입힙니다.
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval() # 평가 모드 전환 (Batch Normalization, Dropout 등의 동작 고정)
        print(f"성공적으로 모델을 불러왔습니다: {model_path}")
    else:
        print("모델 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    # 3. 데이터 로더 준비 (평가 시에는 shuffle=False 권장)
    dataloader = get_dataloader(classes, qualities, train_folder_name, batch_size=32)
    
    # 4. 평가 지표 초기화
    criterion_class = nn.CrossEntropyLoss()
    criterion_quality = nn.CrossEntropyLoss()
    running_loss = 0.0
    correct_class_predictions = 0
    correct_quality_predictions = 0
    total_samples = 0

    print("모델 평가 시작...")
    
    # 5. 성능 측정 (기울기 계산 비활성화)
    with torch.no_grad():
        for images, labels, quali_labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device)
            quali_labels = quali_labels.to(device)

            # 예측값 계산
            out_class, out_quality = model(images)
            loss_class = criterion_class(out_class, labels)
            loss_quality = criterion_class(out_quality, quali_labels)
            total_loss = loss_class + loss_quality
            running_loss += total_loss.item()

            # class의 정확도 계산 (가장 높은 확률값을 가진 인덱스 비교)
            # labels와 out_class는 [batch_size, 5] 형태입니다.
            _, predicted_class_idx = torch.max(out_class, 1)
            _, target_class_idx = torch.max(labels, 1)
            correct_class_predictions += (predicted_class_idx == target_class_idx).sum().item()
            
            # quality의 정확도 계산 (가장 높은 확률값을 가진 인덱스 비교)
            # quali_labels와 out_quality는 [batch_size, 3] 형태입니다.
            _, predicted_quality_idx = torch.max(out_quality, 1)
            _, target_quality_idx = torch.max(quali_labels, 1)
            correct_quality_predictions += (predicted_quality_idx == target_quality_idx).sum().item()
            
            total_samples += labels.size(0)

    # 6. 최종 결과 출력
    avg_loss = running_loss / len(dataloader)
    accuracy_class = (correct_class_predictions / total_samples) * 100
    accuracy_quality = (correct_quality_predictions / total_samples) * 100

    print("\n" + "="*30)
    print(f"Training 데이터 최종 평가 결과")
    print(f"- 평균 손실(Loss): {avg_loss:.4f}")
    print(f"- 과일 종류 정확도: {accuracy_class:.2f}% ({correct_class_predictions}/{total_samples})")
    print(f"- 품질 분류 정확도: {accuracy_quality:.2f}% ({correct_quality_predictions}/{total_samples})")
    print(f"- 전체 샘플 수: {total_samples}")
    print("="*30)

# 실행 예시
if __name__ == "__main__":
    classes = ["apple", "mandarine", "onion", "pear", "potato"]
    qualities = ['super', 'good', 'normal']
    train_folder_name = 'train'
    evaluate_model(classes, qualities, train_folder_name)

평가에 사용하는 장치: cuda
성공적으로 모델을 불러왔습니다: ./models/train_model_5fruits_multitask.pth


/tmp/ipykernel_26621/3635850530.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


모델 평가 시작...


Evaluating:  89%|████████▉ | 2972/3348 [10:36<01:19,  4.70it/s]

# 7. make_test_dataset.py
- 용량이 부족해서 train 학습하고 데이터를 지워야 함(train 폴더의 데이터를 지우자!!)

- test(Validation) 데이터 다운로드 수행

- prompt: 이번에는 Validation에서 위와 동일한 과일들의 데이터를 다운로드 하는 shell 명령어를 작성해줘

(결과물)
aihubshell -mode d -datasetkey 149 -filekey 397011,397012,397013,397014,397015,397016,397023,397024,397025,397029,397030,397031,397032,397033,397034,397035,397036,397037,397038,397039,397040,397041,397044,397045,397046,397056,397057,397058,397059,397060,397061,397043,397042,397077,397078,397079,397083,397084,397085,397086,397087,397088,397089,397090,397091,397092,397093,397094,397095,397096,397097,397098,397099,397100,397110,397111,397112,397113,397114,397115,397065,397066,397067,397068,397069,397070

- 위 명령어를 수행하면, 068.농산물_물질(QC)_이미지 폴더 안에 2.Validation 폴더 안에 데이터가 차곡차곡 쌓일 거임

In [6]:
import os
import zipfile
import json
import shutil
from tqdm import tqdm

# 1. 설정
classes = ["apple", "mandarine", "onion", "pear", "potato"]
# classes = ["apple", "mandarine"]
qualities = ["super", "good", "normal"] # 3단계 등급으로 수정
download_folder_name = r"068.농산물_품질(QC)_이미지/01.데이터/2.Validation/"
train_folder_name = "test"

# 2. 폴더 구조 생성 (train/apple/super, train/apple/good...)
for class_name in classes:
    for q in qualities:
        os.makedirs(os.path.join(train_folder_name, class_name, q), exist_ok=True)

# 3. 데이터 경로 파악 (원천 및 라벨링 폴더 찾기)
dirlist = os.listdir(download_folder_name)
image_base_path = ""
label_base_path = ""

for folder in dirlist:
    if "원천" in folder:
        image_base_path = os.path.join(download_folder_name, folder)
    elif "라벨링" in folder:
        label_base_path = os.path.join(download_folder_name, folder)

# 4. 클래스별 처리 시작
for class_name in classes:
    print(f"\n[{class_name.upper()}] 처리 및 등급 분류 시작...")
    
    # 해당 클래스 이미지 zip과 라벨 json zip 리스트 확보
    image_zips = [z for z in os.listdir(image_base_path) if class_name.lower() in z.lower()]
    label_zips = [z for z in os.listdir(label_base_path) if class_name.lower() in z.lower()]

    # 임시로 압축을 풀 공간 생성
    temp_extract_path = os.path.join(train_folder_name, f"temp_{class_name}")
    os.makedirs(temp_extract_path, exist_ok=True)

    # A. 이미지와 라벨 압축 해제
    for zips, base_p in [(image_zips, image_base_path), (label_zips, label_base_path)]:
        for zip_name in tqdm(zips, desc=f"Extracting {class_name} files"):
            with zipfile.ZipFile(os.path.join(base_p, zip_name), 'r') as zip_ref:
                zip_ref.extractall(temp_extract_path)

    # B. JSON 내용을 보고 이미지 파일 이동 (품질 분류 핵심 로직)
    all_files = os.listdir(temp_extract_path)
    img_files = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for img_name in tqdm(img_files, desc=f"Classifying {class_name} by quality"):
        img_path = os.path.join(temp_extract_path, img_name)
        json_path = os.path.splitext(img_path)[0] + ".json" # 확장자만 json으로 변경

        if os.path.exists(json_path):
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    meta = json.load(f)
                    cate3_val = meta.get('cate3', '')

                    # 등급 매칭 (AI Hub 표준 규격 기준)
                    if cate3_val in ['특']:
                        target_q = "super"
                    elif cate3_val in ['상']:
                        target_q = "good"
                    else: # '보통' 혹은 그 외
                        target_q = "normal"

                    # 파일 이동
                    dst_path = os.path.join(train_folder_name, class_name, target_q, img_name)
                    shutil.move(img_path, dst_path)
            except Exception as e:
                print(f"파일 처리 오류 ({img_name}): {e}")
        
        # 원본 이미지나 JSON이 남아있다면 삭제 (정리)
        if os.path.exists(img_path): os.remove(img_path)
        if os.path.exists(json_path): os.remove(json_path)

    # 임시 폴더 삭제
    shutil.rmtree(temp_extract_path)

print("\n등급별 데이터 구축이 완료되었습니다!")


[APPLE] 처리 및 등급 분류 시작...


Classifying apple by quality: 100%|██████████| 3128/3128 [00:18<00:00, 170.74it/s]



[MANDARINE] 처리 및 등급 분류 시작...


Classifying mandarine by quality: 100%|██████████| 3024/3024 [00:12<00:00, 245.20it/s]



[ONION] 처리 및 등급 분류 시작...


...


[PEAR] 처리 및 등급 분류 시작...


Classifying pear by quality: 100%|██████████| 3024/3024 [00:00<00:00, 20438.52it/s]



[POTATO] 처리 및 등급 분류 시작...


Classifying potato by quality: 100%|██████████| 3024/3024 [00:00<00:00, 21318.22it/s]


등급별 데이터 구축이 완료되었습니다!


## 7_1. Sampling Test Dataset as 500

In [7]:
import os
import random
import shutil

# 0. 샘플링 개수
num_sampling = 500

# 1. 경로 설정
base_dir = os.getcwd()
print(base_dir)
source_root = os.path.join(base_dir, 'test')
target_root = os.path.join(base_dir, f'test_{num_sampling}')

# 2. test_500 루트 폴더 생성
if not os.path.exists(target_root):
    os.makedirs(target_root)

# 3. train 폴더 내의 모든 하위 디렉토리 찾기
subfolders = [f for f in os.listdir(source_root) if os.path.isdir(os.path.join(source_root, f))]

print(f"발견된 폴더 목록: {subfolders}")

for folder in subfolders:
    # 각 폴더의 원본 경로와 복사될 경로 설정
    source_folder_path = os.path.join(source_root, folder)
    target_folder_path = os.path.join(target_root, folder)
    
    # 4. 대상 폴더 안에 동일한 이름의 하위 폴더 생성 (예: train_500/Apple)
    if not os.path.exists(target_folder_path):
        os.makedirs(target_folder_path)
    
    # 5. 해당 폴더 내의 파일 목록 가져오기
    all_files = [f for f in os.listdir(source_folder_path) if os.path.isfile(os.path.join(source_folder_path, f))]
    
    # 6. 랜덤하게 500개 추출 (파일이 500개 미만이면 전체 선택)
    num_to_sample = min(len(all_files), num_sampling)
    sampled_files = random.sample(all_files, num_to_sample)
    
    print(f"[{folder}] 처리 중: {len(all_files)}개 중 {num_to_sample}개 복사 시작...")
    
    # 7. 파일 복사 실행
    for file_name in sampled_files:
        src_file = os.path.join(source_folder_path, file_name)
        dst_file = os.path.join(target_folder_path, file_name)
        shutil.copy2(src_file, dst_file)

print("-" * 30)
print(f"모든 작업이 완료되었습니다. '{target_root}'를 확인하세요!")

/workspace
발견된 폴더 목록: ['onion', 'potato', 'pear', 'apple', 'mandarine']
[onion] 처리 중: 0개 중 0개 복사 시작...
[potato] 처리 중: 0개 중 0개 복사 시작...
[pear] 처리 중: 0개 중 0개 복사 시작...
[apple] 처리 중: 0개 중 0개 복사 시작...
[mandarine] 처리 중: 0개 중 0개 복사 시작...
------------------------------
모든 작업이 완료되었습니다. '/workspace/test_500'를 확인하세요!


`shell`
- 터미널에서 test_500 폴더 안에 들어가서 각 폴더 및 세부 폴더 내 파일 개수를 세는 리눅스 명령어
- find . -maxdepth 2 -type d -exec sh -c "echo -n '{}: '; find '{}' -type f | wc -l" \;

In [8]:
import os
import random
import shutil
from tqdm import tqdm

# 0. 샘플링 개수
num_sampling = 500

# 1. 경로 설정
base_dir = os.getcwd()
source_root = os.path.join(base_dir, 'test')
target_root = os.path.join(base_dir, f'test_{num_sampling}')

print(f"작업 시작: {source_root} -> {target_root}")

# 2. os.walk를 이용해 모든 하위 폴더 순회
# root: 현재 탐색 중인 폴더 경로
# dirs: 현재 폴더 안의 하위 폴더 리스트
# files: 현재 폴더 안의 파일 리스트
for root, dirs, files in os.walk(source_root):
    
    # 3. 대상 폴더(target_root) 내에 동일한 상대 경로 구조 생성
    # source_root로부터의 상대 경로를 구함 (예: apple/good)
    rel_path = os.path.relpath(root, source_root)
    target_folder_path = os.path.join(target_root, rel_path)
    
    # 폴더 생성 (이미 있으면 무시)
    os.makedirs(target_folder_path, exist_ok=True)
    
    # 4. 현재 폴더에 파일이 있는 경우에만 샘플링 수행
    if files:
        # 파일들 중 실제 파일인 것만 필터링 (간혹 폴더가 섞일 수 있음)
        all_files = [f for f in files if os.path.isfile(os.path.join(root, f))]
        
        # 5. 랜덤하게 num_sampling개 추출 (파일이 부족하면 전체 선택)
        num_to_sample = min(len(all_files), num_sampling)
        sampled_files = random.sample(all_files, num_to_sample)
        
        # 6. 파일 복사 실행
        desc = f"복사 중: {rel_path if rel_path != '.' else 'root'}"
        for file_name in tqdm(sampled_files, desc=desc, leave=False):
            src_file = os.path.join(root, file_name)
            dst_file = os.path.join(target_folder_path, file_name)
            shutil.copy2(src_file, dst_file)

print("-" * 30)
print(f"모든 작업이 완료되었습니다.")
print(f"결과 경로: {target_root}")

작업 시작: /workspace/test -> /workspace/test_500


------------------------------
모든 작업이 완료되었습니다.
결과 경로: /workspace/test_500


# 8. test.py

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm
# 이전에 정의한 클래스와 함수들을 가져옵니다.
# from model import get_model
# from dataloader import get_dataloader

def evaluate_model(classes, qualities, train_folder_name, model_path='./models/train_model_5fruits_multitask.pth'):
    # 1. 장치 설정
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"평가에 사용하는 장치: {device}")

    # 2. 모델 초기화 및 가중치 불러오기
    model = get_model(classes, qualities)
    
    if os.path.exists(model_path):
        # 저장된 state_dict를 불러와 모델에 입힙니다.
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval() # 평가 모드 전환 (Batch Normalization, Dropout 등의 동작 고정)
        print(f"성공적으로 모델을 불러왔습니다: {model_path}")
    else:
        print("모델 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    # 3. 데이터 로더 준비 (평가 시에는 shuffle=False 권장)
    dataloader = get_dataloader(classes, qualities, train_folder_name, batch_size=32)
    
    # 4. 평가 지표 초기화
    criterion_class = nn.CrossEntropyLoss()
    criterion_quality = nn.CrossEntropyLoss()
    running_loss = 0.0
    correct_class_predictions = 0
    correct_quality_predictions = 0
    total_samples = 0

    print("모델 평가 시작...")
    
    # 5. 성능 측정 (기울기 계산 비활성화)
    with torch.no_grad():
        for images, labels, quali_labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device)
            quali_labels = quali_labels.to(device)

            # 예측값 계산
            out_class, out_quality = model(images)
            loss_class = criterion_class(out_class, labels)
            loss_quality = criterion_class(out_quality, quali_labels)
            total_loss = loss_class + loss_quality
            running_loss += total_loss.item()

            # class의 정확도 계산 (가장 높은 확률값을 가진 인덱스 비교)
            # labels와 out_class는 [batch_size, 5] 형태입니다.
            _, predicted_class_idx = torch.max(out_class, 1)
            _, target_class_idx = torch.max(labels, 1)
            correct_class_predictions += (predicted_class_idx == target_class_idx).sum().item()
            
            # quality의 정확도 계산 (가장 높은 확률값을 가진 인덱스 비교)
            # quali_labels와 out_quality는 [batch_size, 3] 형태입니다.
            _, predicted_quality_idx = torch.max(out_quality, 1)
            _, target_quality_idx = torch.max(quali_labels, 1)
            correct_quality_predictions += (predicted_quality_idx == target_quality_idx).sum().item()
            
            total_samples += labels.size(0)

    # 6. 최종 결과 출력
    avg_loss = running_loss / len(dataloader)
    accuracy_class = (correct_class_predictions / total_samples) * 100
    accuracy_quality = (correct_quality_predictions / total_samples) * 100

    print("\n" + "="*30)
    print(f"Training 데이터 최종 평가 결과")
    print(f"- 평균 손실(Loss): {avg_loss:.4f}")
    print(f"- 과일 종류 정확도: {accuracy_class:.2f}% ({correct_class_predictions}/{total_samples})")
    print(f"- 품질 분류 정확도: {accuracy_quality:.2f}% ({correct_quality_predictions}/{total_samples})")
    print(f"- 전체 샘플 수: {total_samples}")
    print("="*30)

# 실행 예시
if __name__ == "__main__":
    classes = ["apple", "mandarine", "onion", "pear", "potato"]
    qualities = ['super', 'good', 'normal']
    train_folder_name = 'test_500'
    evaluate_model(classes, qualities, train_folder_name)

In [9]:
# import torch
# import torch.nn as nn
# from tqdm import tqdm
# # 이전에 정의한 클래스와 함수들을 가져옵니다.
# # from model import get_model
# # from dataloader import get_dataloader

# def evaluate_model(classes, test_folder_name, model_path='./models/train_model_5fruits.pth'):
#     # 1. 장치 설정
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"평가에 사용하는 장치: {device}")

#     # 2. 모델 초기화 및 가중치 불러오기
#     model = get_model(classes)
    
#     if os.path.exists(model_path):
#         # 저장된 state_dict를 불러와 모델에 입힙니다.
#         model.load_state_dict(torch.load(model_path, map_location=device))
#         model.to(device)
#         model.eval() # 평가 모드 전환 (Batch Normalization, Dropout 등의 동작 고정)
#         print(f"성공적으로 모델을 불러왔습니다: {model_path}")
#     else:
#         print("모델 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
#         return

#     # 3. 데이터 로더 준비 (평가 시에는 shuffle=False 권장)
#     dataloader = get_dataloader(classes, test_folder_name, batch_size=32)
    
#     # 4. 평가 지표 초기화
#     criterion = nn.BCELoss()
#     running_loss = 0.0
#     correct_predictions = 0
#     total_samples = 0

#     print("모델 평가 시작...")
    
#     # 5. 성능 측정 (기울기 계산 비활성화)
#     with torch.no_grad():
#         for images, labels in tqdm(dataloader, desc="Evaluating"):
#             images = images.to(device)
#             labels = labels.to(device)

#             # 예측값 계산
#             outputs = model(images)
#             loss = criterion(outputs, labels)
#             running_loss += loss.item()

#             # 정확도 계산 (가장 높은 확률값을 가진 인덱스 비교)
#             # labels와 outputs는 [batch_size, 5] 형태입니다.
#             _, predicted_idx = torch.max(outputs, 1)
#             _, target_idx = torch.max(labels, 1)
            
#             correct_predictions += (predicted_idx == target_idx).sum().item()
#             total_samples += labels.size(0)

#     # 6. 최종 결과 출력
#     avg_loss = running_loss / len(dataloader)
#     accuracy = (correct_predictions / total_samples) * 100

#     print("\n" + "="*30)
#     print(f"Validation 데이터 최종 평가 결과")
#     print(f"- 평균 손실(Loss): {avg_loss:.4f}")
#     print(f"- 정확도(Accuracy): {accuracy:.2f}%")
#     print(f"- 전체 샘플 수: {total_samples}")
#     print("="*30)

# # 실행 예시
# if __name__ == "__main__":
#     classes = ["apple", "mandarine", "onion", "pear", "potato"]
#     test_folder_name = 'test'
#     evaluate_model(classes, test_folder_name)

/tmp/ipykernel_6630/2038579607.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


평가에 사용하는 장치: cuda
성공적으로 모델을 불러왔습니다: ./models/train_model_5fruits.pth
모델 평가 시작...


Evaluating: 100%|██████████| 482/482 [01:42<00:00,  4.72it/s]


최종 평가 결과
- 평균 손실(Loss): 0.1311
- 정확도(Accuracy): 92.89%
- 전체 샘플 수: 15400


# 9. train_model_5fruits 모델 평가 및 분석

모델 상태 진단 및 분석
진단: 과대적합 (Overfitting)

근거: 훈련 데이터 정확도(99.62%)는 매우 높지만, 테스트 데이터 정확도(92.89%)와 약 7% 정도의 큰 격차가 발생합니다. 또한 훈련 손실(0.0062) 대비 테스트 손실(0.1311)이 약 20배 이상 높습니다. 

원인 분석:


모델의 단순함: 레이어가 3개로 얕고 필터 수가 적어 데이터의 복잡한 특징을 일반화하기보다 특정 이미지를 통째로 외웠을 가능성이 큽니다. 


규제(Regularization) 부재: 드롭아웃(Dropout)이나 배치 정규화(Batch Normalization) 등 과대적합을 방지할 수 있는 장치가 전혀 없습니다. 


데이터 다양성 부족 대비 과한 학습: 단 2에폭만으로 훈련 정확도가 100%에 근접했다는 것은 모델이 데이터의 핵심 특징보다는 지엽적인 정보를 빠르게 학습했음을 의미합니다.

# 10. 모델 아키텍처 업그레이드

- 아키텍처 업그레이드

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FruitClassifier(nn.Module):
    def __init__(self, num_classes):
        super(FruitClassifier, self).__init__()
        
        # 레이어 1: 필터 수를 16 -> 32로 확장 + BatchNorm 추가
        self.conv1 = nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2)
        self.bn1 = nn.BatchNorm2d(32)
        
        # 레이어 2: 필터 수 32 -> 64로 확장 + BatchNorm 추가
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # 레이어 3: 깊이 추가 (64 -> 128) + BatchNorm 추가
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # 레이어 4: 고차원 특징 추출을 위한 레이어 추가
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        
        # 전역 평균 풀링
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # 과대적합 방지를 위한 Dropout 추가 (50% 확률)
        self.dropout = nn.Dropout(0.5)
        
        # 최종 분류 레이어 (입력 크기가 128로 증가)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # 레이어 1
        x = F.silu(self.bn1(self.conv1(x)))
        # 레이어 2
        x = F.silu(self.bn2(self.conv2(x)))
        # 레이어 3
        x = F.silu(self.bn3(self.conv3(x)))
        # 레이어 4
        x = F.silu(self.bn4(self.conv4(x)))
        
        x = self.gap(x)
        x = x.view(x.size(0), -1) # 평탄화
        
        # 출력 전 Dropout 적용
        x = self.dropout(x)
        
        # 최종 확률 반환
        x = F.softmax(self.fc(x), dim=1)
        
        return x

def get_model(classes):
    num_classes = len(classes)
    model = FruitClassifier(num_classes)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    return model

# 11. 모델 재학습

- 저장한 모델 이름: train_model_5fruits_upgrade.pth

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FruitClassifier(nn.Module):
    def __init__(self, num_classes):
        super(FruitClassifier, self).__init__()
        
        # 레이어 1: 기초 특징 추출 (직선, 곡선 등)
        # In: (N, 3, 64, 64) -> Out: (N, 16, 32, 32)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=2, padding=2)
        
        # 레이어 2: 복잡한 특징 추출
        # In: (N, 16, 32, 32) -> Out: (N, 32, 16, 16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=2, padding=1)
        
        # 레이어 3: 더 복잡한 특징 추출
        # In: (N, 32, 16, 16) -> Out: (N, 32, 8, 8)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=2, padding=1)
        
        # 전역 평균 풀링 (Global Average Pooling)
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # 최종 분류 레이어 (확률 반환)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        # x: [Batch, Channel, Height, Width]
        
        x = F.silu(self.conv1(x)) # SiLU는 TensorFlow의 Swish와 동일한 함수입니다.
        x = F.silu(self.conv2(x))
#         x = F.silu(self.conv3(x))
        
        x = self.gap(x)           # (N, 32, 1, 1) 형태로 변환
        x = x.view(x.size(0), -1) # (N, 32)로 평탄화
        
        # PyTorch의 CrossEntropyLoss를 사용할 경우 마지막에 Softmax를 생략하는 것이 표준이지만,
        # 기존 설계대로 확률값을 반환하기 위해 Softmax를 명시합니다.
        x = F.softmax(self.fc(x), dim=1)
        
        return x

def get_model(classes):
    num_classes = len(classes)
    model = FruitClassifier(num_classes)
    
    # GPU(H100) 환경이 감지되면 모델을 GPU로 이동시킵니다.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    return model

In [13]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm  # 학습 진행률 표시를 위한 라이브러리

# 앞서 정의한 get_model과 get_dataloader가 정의되어 있어야 합니다.
# from model import get_model
# from dataloader import get_dataloader

# 1. 장치 설정 (H100 GPU 사용 가능 여부 확인)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 학습에 사용하는 장치: {device}")

# 2. 클래스 및 폴더 설정
classes = ["apple", "mandarine", "onion", "pear", "potato"]
train_folder_name = 'train'

if not os.path.exists('./models'):
    os.makedirs('./models')

# 3. 모델 및 데이터 로더 초기화
model = get_model(classes)  # 이미 내부에서 .to(device) 처리가 됨
dataloader = get_dataloader(classes, train_folder_name, batch_size=32)

# 4. 손실 함수 및 최적화 함수 설정
# TensorFlow의 categorical_crossentropy는 PyTorch의 CrossEntropyLoss와 매칭됩니다.
# (라벨이 One-hot 형태인 경우 BCEWithLogitsLoss가 더 적합할 수 있으나, 기존 설계를 따릅니다.)
criterion = nn.BCELoss() # 모델의 마지막이 Softmax이므로 Binary Cross Entropy 사용
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. 학습 루프 정의 (TensorFlow의 model.fit 역할)
num_epochs = 2
best_loss = float('inf')

for epoch in range(num_epochs):
    model.train()  # 모델을 학습 모드로 설정
    running_loss = 0.0
    
    # tqdm을 사용하여 진행률 표시
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, labels in pbar:
        # 데이터를 GPU로 이동
        images = images.to(device)
        labels = labels.to(device)
        
        # 가중치 초기화
        optimizer.zero_grad()
        
        # 순전파 (Forward)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 역전파 및 가중치 업데이트 (Backward & Optimize)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': running_loss / (pbar.n + 1)})

    # 에폭 종료 후 평균 손실 계산
    epoch_loss = running_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Average Loss: {epoch_loss:.4f}")

    # ModelCheckpoint 기능 구현 (가장 낮은 loss일 때 저장)
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), './models/train_model_5fruits_upgrade2_epoch2.pth')
        print(f"  --> Best model saved with loss: {best_loss:.4f}")

print("학습이 완료되었습니다.")

현재 학습에 사용하는 장치: cuda


Epoch 1/2: 100%|██████████| 3350/3350 [11:46<00:00,  4.74it/s, loss=0.0632]


Epoch [1/2] Average Loss: 0.0631
  --> Best model saved with loss: 0.0631


Epoch 2/2: 100%|██████████| 3350/3350 [11:46<00:00,  4.74it/s, loss=0.0224]

Epoch [2/2] Average Loss: 0.0224
  --> Best model saved with loss: 0.0224
학습이 완료되었습니다.


# 12. 모델 평가하기 using train


In [16]:
# for train_800 이용 모델 평가

import torch
import torch.nn as nn
from tqdm import tqdm
# 이전에 정의한 클래스와 함수들을 가져옵니다.
# from model import get_model
# from dataloader import get_dataloader

def evaluate_model(classes, train_folder_name, model_path):
    # 1. 장치 설정
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"평가에 사용하는 장치: {device}")

    # 2. 모델 초기화 및 가중치 불러오기
    model = get_model(classes)
    
    if os.path.exists(model_path):
        # 저장된 state_dict를 불러와 모델에 입힙니다.
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval() # 평가 모드 전환 (Batch Normalization, Dropout 등의 동작 고정)
        print(f"성공적으로 모델을 불러왔습니다: {model_path}")
    else:
        print("모델 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    # 3. 데이터 로더 준비 (평가 시에는 shuffle=False 권장)
    dataloader = get_dataloader(classes, train_folder_name, batch_size=32)
    
    # 4. 평가 지표 초기화
    criterion = nn.BCELoss()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    print("모델 평가 시작...")
    
    # 5. 성능 측정 (기울기 계산 비활성화)
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device)

            # 예측값 계산
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # 정확도 계산 (가장 높은 확률값을 가진 인덱스 비교)
            # labels와 outputs는 [batch_size, 5] 형태입니다.
            _, predicted_idx = torch.max(outputs, 1)
            _, target_idx = torch.max(labels, 1)
            
            correct_predictions += (predicted_idx == target_idx).sum().item()
            total_samples += labels.size(0)

    # 6. 최종 결과 출력
    avg_loss = running_loss / len(dataloader)
    accuracy = (correct_predictions / total_samples) * 100

    print("\n" + "="*30)
    print(f"Training 데이터 최종 평가 결과")
    print(f"- 평균 손실(Loss): {avg_loss:.4f}")
    print(f"- 정확도(Accuracy): {accuracy:.2f}%")
    print(f"- 전체 샘플 수: {total_samples}")
    print("="*30)

# 실행 예시
if __name__ == "__main__":
    classes = ["apple", "mandarine", "onion", "pear", "potato"]
    train_folder_name = 'train'
    model_name = './models/train_model_5fruits_upgrade2_epoch2.pth'
    evaluate_model(classes, train_folder_name, model_name)

평가에 사용하는 장치: cuda
성공적으로 모델을 불러왔습니다: ./models/train_model_5fruits_upgrade2_epoch2.pth


/tmp/ipykernel_24516/1891539698.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


모델 평가 시작...


Evaluating: 100%|██████████| 3350/3350 [11:45<00:00,  4.75it/s]


Training 데이터 최종 평가 결과
- 평균 손실(Loss): 0.0192
- 정확도(Accuracy): 98.58%
- 전체 샘플 수: 107200


# 13. 모델 평가하기 using test_500(Validation 데이터)

In [14]:
# for test 이용(Validation data) 모델 평가

import torch
import torch.nn as nn
from tqdm import tqdm
# 이전에 정의한 클래스와 함수들을 가져옵니다.
# from model import get_model
# from dataloader import get_dataloader

def evaluate_model(classes, test_folder_name, model_path):
    # 1. 장치 설정
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"평가에 사용하는 장치: {device}")

    # 2. 모델 초기화 및 가중치 불러오기
    model = get_model(classes)
    
    if os.path.exists(model_path):
        # 저장된 state_dict를 불러와 모델에 입힙니다.
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval() # 평가 모드 전환 (Batch Normalization, Dropout 등의 동작 고정)
        print(f"성공적으로 모델을 불러왔습니다: {model_path}")
    else:
        print("모델 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    # 3. 데이터 로더 준비 (평가 시에는 shuffle=False 권장)
    dataloader = get_dataloader(classes, test_folder_name, batch_size=32)
    
    # 4. 평가 지표 초기화
    criterion = nn.BCELoss()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    print("모델 평가 시작...")
    
    # 5. 성능 측정 (기울기 계산 비활성화)
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device)

            # 예측값 계산
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            # 정확도 계산 (가장 높은 확률값을 가진 인덱스 비교)
            # labels와 outputs는 [batch_size, 5] 형태입니다.
            _, predicted_idx = torch.max(outputs, 1)
            _, target_idx = torch.max(labels, 1)
            
            correct_predictions += (predicted_idx == target_idx).sum().item()
            total_samples += labels.size(0)

    # 6. 최종 결과 출력
    avg_loss = running_loss / len(dataloader)
    accuracy = (correct_predictions / total_samples) * 100

    print("\n" + "="*30)
    print(f"Validation 데이터 최종 평가 결과")
    print(f"- 평균 손실(Loss): {avg_loss:.4f}")
    print(f"- 정확도(Accuracy): {accuracy:.2f}%")
    print(f"- 전체 샘플 수: {total_samples}")
    print("="*30)

# 실행 예시
if __name__ == "__main__":
    classes = ["apple", "mandarine", "onion", "pear", "potato"]
    test_folder_name = 'test_500'
    model_name = './models/train_model_5fruits_upgrade2_epoch2.pth'
    evaluate_model(classes, test_folder_name, model_name)

/tmp/ipykernel_24516/2311986663.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


평가에 사용하는 장치: cuda
성공적으로 모델을 불러왔습니다: ./models/train_model_5fruits_upgrade2_epoch2.pth
모델 평가 시작...


Evaluating: 100%|██████████| 79/79 [00:17<00:00,  4.58it/s]


Validation 데이터 최종 평가 결과
- 평균 손실(Loss): 0.3561
- 정확도(Accuracy): 86.96%
- 전체 샘플 수: 2500


# 평가
- 이전 학습 모델보다 test 결과가 더 떨어짐.
- 이유는?


만약 train보다 test 점수가 10 이상 차이가 난다면?

훈련 데이터(Train)보다 테스트 데이터(Test) 점수가 10% 이상 높게 나온다면, 이는 단순한 오차 범위를 벗어난 아주 특이한 상황입니다. 이 정도의 큰 격차는 모델 학습의 문제라기보다 데이터셋의 설계나 평가 방식의 근본적인 문제일 확률이 매우 높습니다.

다음 4가지 시나리오로 분석하고 해결책을 제시해 드립니다.

1. 데이터 분포의 불균형 (가장 유력한 원인)
학습 데이터셋은 매우 어렵고 복잡한 이미지들로 구성된 반면, 테스트 데이터셋은 모델이 맞히기 너무 쉬운 "꿀" 데이터들로만 구성된 경우입니다.

분석: 예를 들어, Train 세트에는 어둡고 흔들린 과일 사진이 많은데, Test 세트에는 조명이 밝은 스튜디오 사진만 있는 경우입니다.

해결책: 데이터를 다시 무작위로 섞어서(Shuffle) Train/Test 비율을 다시 나누어야 합니다. (예: 8:2 또는 7:3)

2. 테스트 데이터셋의 크기가 너무 작음
테스트 데이터의 샘플 수가 너무 적으면(예: 수십 장 내외), 모델이 몇 개만 우연히 더 맞춰도 점수가 10% 이상 널뛰게 됩니다.

분석: 통계적 유의성이 부족하여 발생하는 수치적 착시 현상입니다.

해결책: 테스트 데이터의 양을 전체 데이터의 최소 15~20% 이상으로 확보하세요.

3. 데이터 유출 (Data Leakage)의 역방향 가능성
드문 경우지만, 테스트 데이터의 정보가 어떤 식으로든 학습 과정에 유리하게 작용했을 수 있습니다. 혹은 중복된 데이터가 테스트 세트에만 대량으로 포함되어 있을 때 발생합니다.

분석: 테스트 세트 내에 동일하거나 매우 유사한 이미지가 반복적으로 들어있어 점수가 뻥튀기된 상태입니다.

해결책: 데이터셋에 중복된 파일이 없는지, 파일명이 다른데 같은 이미지인 파일이 있는지 전수 조사가 필요합니다.

4. 강력한 드롭아웃(Dropout) 효과
모델 아키텍처에서 Dropout 비율을 0.7~0.8 정도로 극단적으로 높게 설정했을 때 발생할 수 있습니다.

분석: 학습 때는 모델의 손발을 너무 심하게 묶어놓아서 빌빌대다가, 테스트 때(model.eval()) 손발을 다 풀어주니 갑자기 날아다니는 경우입니다.

해결책: 드롭아웃 수치를 0.3~0.5 정도로 낮추어 학습과 테스트 간의 간극을 줄여주세요.

📊 수치별 진단 요약
격차 정도	진단	위험도
1~3% 높음	아주 이상적이고 건강한 일반화 상태	매우 낮음 (권장)
5% 내외 높음	모델에 적용된 규제(Dropout 등)가 매우 강력함	중간 (학습 더 필요)
10% 이상 높음	데이터셋 분할 오류 또는 샘플 부족	매우 높음 (재설계 필요)
🛠 조치 순서 제안
데이터 재분할: train_test_split 시 shuffle=True 옵션을 주고 다시 데이터를 나눈 뒤 학습해 보세요.

교차 검증(Cross-Validation): 데이터를 5개 그룹으로 나눠 돌려가며 검증해 보세요. 모든 그룹에서 Test가 높게 나오는지 확인해야 합니다.

랜덤 샘플 확인: 모델이 틀린 이미지와 맞힌 이미지를 눈으로 직접 확인(Visualization)하여, 특정 환경의 이미지만 Test에 몰려 있는지 체크하세요.